# RAG & Vector Databases
**Retrieval-Augmented Generation** with real embeddings and vector search.

**Prerequisites:**
```bash
pip install sentence-transformers faiss-cpu chromadb numpy matplotlib scikit-learn
```

## Cell 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.metrics.pairwise import cosine_similarity
import time

np.random.seed(42)

## Section 1: Real Embeddings with Sentence-Transformers

In [ ]:
# Load a small, fast embedding model
print('Loading sentence-transformers model...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded successfully!')

# Encode real sentences
sentences = [
    "Python is a popular programming language for data science.",
    "Neural networks are inspired by biological neurons.",
    "Transformers use self-attention for sequence modeling.",
    "RAG combines retrieval with generation for better answers.",
    "Vector databases store and search embeddings efficiently.",
    "Fine-tuning adapts a pre-trained model to a specific task.",
    "Gradient descent minimizes the loss function iteratively.",
    "Convolutional layers detect spatial patterns in images.",
    "Recurrent networks process sequential data like text.",
    "Reinforcement learning trains agents through rewards.",
]

embeddings = model.encode(sentences)
print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"\nSample embedding (first 10 dims): {embeddings[0][:10]}")

## Section 2: Cosine Similarity with Real Embeddings
Semantically similar sentences should show high similarity scores.

In [ ]:
# Compute cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)

# Set labels with shorter text for readability
labels = ['Python', 'NeuralNets', 'Transformers', 'RAG', 'VectorDB', 'FineTune', 'GradDescent', 'Conv', 'RNN', 'RL']
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)

# Add similarity values
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7, color='black' if sim_matrix[i,j] < 0.5 else 'white')

plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.title('Cosine Similarity Matrix of Real Embeddings', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print some insights
print("\nHighest similarity pairs:")
similarities = []
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        similarities.append((sim_matrix[i,j], i, j))
similarities.sort(reverse=True)
for score, i, j in similarities[:5]:
    print(f"  {score:.4f}: '{labels[i]}' <-> '{labels[j]}'")

## Section 3: FAISS Vector Index
Build a fast approximate nearest neighbor index with FAISS.

In [ ]:
# Prepare embeddings for FAISS
embeddings_np = embeddings.copy().astype('float32')
dim = embeddings_np.shape[1]

# Build FAISS index
index = faiss.IndexFlatIP(dim)  # Inner product (cosine sim for normalized vecs)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings_np)
index.add(embeddings_np)

print(f"FAISS index created with {index.ntotal} vectors of dimension {dim}")

# Query with a new sentence
query_text = "How do neural networks learn?"
query_emb = model.encode([query_text]).astype('float32')
faiss.normalize_L2(query_emb)

D, I = index.search(query_emb, k=3)

print(f"\nQuery: '{query_text}'")
print(f"\nTop-3 retrieved documents:")
for rank, (idx, score) in enumerate(zip(I[0], D[0])):
    print(f"  {rank+1}. [{score:.4f}] {sentences[idx]}")

## Section 4: RAG Pipeline with Real Embeddings
A complete RAG pipeline that embeds questions and retrieves relevant documents.

In [ ]:
# Knowledge base of ML facts
knowledge_base = [
    "Transformers revolutionized NLP by using multi-head self-attention mechanisms.",
    "BERT is trained with masked language modeling and next sentence prediction.",
    "GPT models use decoder-only architecture for text generation.",
    "Attention is all you need - the transformer paper introduced self-attention.",
    "Vector embeddings capture semantic relationships between words and concepts.",
    "Cosine similarity measures the angle between two embedding vectors.",
    "RAG reduces hallucination by grounding generation in retrieved documents.",
    "FAISS enables billion-scale similarity search for vector databases.",
    "Chunking strategies affect retrieval quality - size and overlap matter.",
    "Fine-tuning adapts pre-trained models to downstream tasks with labeled data.",
    "Cross-encoders rerank retrieved documents for better precision.",
    "Sparse retrievers use BM25, dense retrievers use learned embeddings.",
    "Hybrid search combines sparse and dense methods for robust retrieval.",
    "Token limit in context windows constrains the number of retrieved documents.",
    "Evaluation metrics for retrieval include MRR, NDCG, Precision@k, and Recall@k.",
    "Embedding models matter as much as LLM choice for RAG performance.",
    "Contrastive learning trains embeddings by minimizing positive pair distance.",
    "Vector databases like Pinecone and Weaviate manage embeddings at scale.",
]

# Encode knowledge base
kb_embeddings = model.encode(knowledge_base).astype('float32')
faiss.normalize_L2(kb_embeddings)

# Build KB index
kb_index = faiss.IndexFlatIP(kb_embeddings.shape[1])
kb_index.add(kb_embeddings)

print(f"Knowledge base loaded with {len(knowledge_base)} facts.\n")

def rag_pipeline(question, k=3):
    """Complete RAG pipeline: embed -> retrieve -> augment -> ready for generation."""
    # Step 1: Embed the question
    q_emb = model.encode([question]).astype('float32')
    faiss.normalize_L2(q_emb)
    
    # Step 2: Retrieve top-k documents
    D, I = kb_index.search(q_emb, k=k)
    retrieved_docs = [knowledge_base[i] for i in I[0]]
    
    # Step 3: Construct augmented prompt
    context = '\n'.join([f"- {doc}" for doc in retrieved_docs])
    augmented_prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer based on the context above:"
    )
    
    return {
        'question': question,
        'retrieved_docs': list(zip(retrieved_docs, D[0])),
        'augmented_prompt': augmented_prompt
    }

# Demo RAG pipeline
test_question = "What is RAG and why is it useful?"
result = rag_pipeline(test_question, k=3)

print(f"QUESTION: {result['question']}")
print(f"\nRETRIEVED DOCUMENTS:")
for i, (doc, score) in enumerate(result['retrieved_docs'], 1):
    print(f"  {i}. [{score:.4f}] {doc}")
print(f"\nAUGMENTED PROMPT (ready for LLM):")
print(result['augmented_prompt'])
print("\n[LLM would generate answer here based on the augmented prompt]")

## Section 5: Chunking Strategies
Large documents must be split into chunks before embedding and retrieval.

In [ ]:
def fixed_size_chunks(text, chunk_size=100, overlap=20):
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def semantic_chunks(text, model, min_chunk_size=50):
    """Split text into sentences and group by semantic similarity."""
    sentences = text.replace('!', '.').replace('?', '.').split('.')
    sentences = [s.strip() for s in sentences if len(s.strip()) > min_chunk_size]
    return sentences

# Example document
sample_doc = (
    "Transformers have become the dominant architecture in modern NLP. "
    "They use self-attention to process all tokens in parallel, unlike RNNs. "
    "BERT is a bidirectional transformer trained with masked language modeling. "
    "GPT models are unidirectional decoders used for text generation. "
    "Fine-tuning these models adapts them to specific downstream tasks. "
    "Vector embeddings from these models capture semantic meaning effectively. "
    "RAG systems retrieve relevant documents and augment prompts for generation. "
    "Chunking strategies affect both retrieval quality and computational cost."
)

# Show chunking strategies
fixed_chunks = fixed_size_chunks(sample_doc, chunk_size=80, overlap=20)
semantic_chunks_result = semantic_chunks(sample_doc, model)

print(f"Document length: {len(sample_doc)} characters")
print(f"\nFixed-size chunks (chunk_size=80, overlap=20):")
for i, chunk in enumerate(fixed_chunks, 1):
    print(f"  Chunk {i} ({len(chunk)} chars): {chunk[:50]}...")

print(f"\nSemantic chunks (sentence-based):")
for i, chunk in enumerate(semantic_chunks_result, 1):
    print(f"  Chunk {i} ({len(chunk)} chars): {chunk[:60]}...")

# Visualize chunk overlap
fig, ax = plt.subplots(figsize=(12, 2))
colors = plt.cm.Set3(np.linspace(0, 1, len(fixed_chunks)))
for i, chunk in enumerate(fixed_chunks):
    start = i * (80 - 20)
    ax.barh(0, len(chunk), left=start, height=0.6, alpha=0.7, color=colors[i], label=f'Chunk {i+1}')
ax.set_xlabel('Character Position', fontsize=10)
ax.set_title('Fixed-Size Chunks with 20-Character Overlap', fontsize=11, fontweight='bold')
ax.set_yticks([])
ax.legend(loc='upper right', ncol=len(fixed_chunks), fontsize=8)
plt.tight_layout()
plt.show()

## Section 6: FAISS Index Types Comparison
Compare IndexFlatIP vs IndexIVFFlat on speed and accuracy with larger datasets.

In [ ]:
# Generate 10K random embeddings to benchmark index types
print("Generating 10,000 random embeddings...")
n_vectors = 10000
dim = embeddings_np.shape[1]

# Create random normalized embeddings
large_embeddings = np.random.randn(n_vectors, dim).astype('float32')
faiss.normalize_L2(large_embeddings)

# Prepare query
query_emb = np.random.randn(1, dim).astype('float32')
faiss.normalize_L2(query_emb)

# Test 1: IndexFlatIP (exact, linear scan)
print("\n1. IndexFlatIP (Exact/Brute-Force):")
index_flat = faiss.IndexFlatIP(dim)
index_flat.add(large_embeddings)

start = time.time()
D_flat, I_flat = index_flat.search(query_emb, k=10)
time_flat = time.time() - start
print(f"   Search time: {time_flat*1000:.2f} ms")
print(f"   Complexity: O(n) = O({n_vectors})")

# Test 2: IndexIVFFlat (approximate, with clustering)
print("\n2. IndexIVFFlat (Approximate with IVF):")
nlist = 100  # number of clusters
index_ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, nlist)
index_ivf.train(large_embeddings)
index_ivf.add(large_embeddings)
index_ivf.nprobe = 10  # number of clusters to search

start = time.time()
D_ivf, I_ivf = index_ivf.search(query_emb, k=10)
time_ivf = time.time() - start
print(f"   Search time: {time_ivf*1000:.2f} ms")
print(f"   Complexity: O(n_clusters + local_search) ≈ O({n_vectors//nlist})")
print(f"   Speed-up: {time_flat/time_ivf:.1f}x faster")

# Calculate recall (what % of top-10 exact matches are in approximate results)
exact_top_10 = set(I_flat[0])
approx_top_10 = set(I_ivf[0])
overlap = len(exact_top_10 & approx_top_10)
recall = overlap / len(exact_top_10)
print(f"   Recall@10: {recall:.2%}")

# Visualization: Speed vs Accuracy tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Speed comparison
methods = ['IndexFlatIP\n(Exact)', 'IndexIVFFlat\n(Approximate)']
times = [time_flat*1000, time_ivf*1000]
colors_bar = ['#FF6B6B', '#4ECDC4']
ax1.bar(methods, times, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Search Time (ms)', fontsize=10)
ax1.set_title('Speed Comparison', fontsize=11, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for i, (m, t) in enumerate(zip(methods, times)):
    ax1.text(i, t + 0.01, f'{t:.2f}ms', ha='center', fontsize=9, fontweight='bold')

# Accuracy vs Speed tradeoff
nprobe_values = [5, 10, 20, 50, 100]
recalls = []
search_times = []
for np_val in nprobe_values:
    index_ivf.nprobe = np_val
    start = time.time()
    D_temp, I_temp = index_ivf.search(query_emb, k=10)
    search_times.append((time.time() - start)*1000)
    overlap_temp = len(set(I_flat[0]) & set(I_temp[0]))
    recalls.append(overlap_temp / len(I_flat[0]))

ax2.plot(search_times, recalls, marker='o', markersize=8, linewidth=2, color='#4ECDC4')
ax2.set_xlabel('Search Time (ms)', fontsize=10)
ax2.set_ylabel('Recall@10', fontsize=10)
ax2.set_title('Speed-Accuracy Tradeoff', fontsize=11, fontweight='bold')
ax2.grid(alpha=0.3)
for x, y, nprobe in zip(search_times, recalls, nprobe_values):
    ax2.text(x, y+0.02, f'nprobe={nprobe}', fontsize=8)

plt.tight_layout()
plt.show()

print("\nKey insights:")
print(f"  - IndexFlatIP: Guaranteed exact results, but slower for large datasets")
print(f"  - IndexIVFFlat: Much faster, with tunable recall via nprobe parameter")
print(f"  - For {n_vectors:,} vectors, IVF is {time_flat/time_ivf:.0f}x faster at nprobe=10")

## Section 7: Evaluation Metrics for Retrieval
Common metrics: Precision@k, Recall@k, MRR, NDCG.

In [ ]:
def precision_at_k(retrieved, relevant, k):
    """Precision@k: fraction of retrieved docs that are relevant."""
    if k == 0:
        return 0.0
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    """Recall@k: fraction of relevant docs that are retrieved."""
    if len(relevant) == 0:
        return 0.0
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def mean_reciprocal_rank(retrieved, relevant):
    """MRR: reciprocal of rank of first relevant document."""
    for i, doc in enumerate(retrieved):
        if doc in relevant:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved_scores, relevant, k):
    """NDCG@k: normalized discounted cumulative gain."""
    # DCG: sum of (relevance / log(rank+1))
    dcg = 0.0
    for i, (doc, score) in enumerate(retrieved_scores[:k]):
        relevance = 1.0 if doc in relevant else 0.0
        dcg += relevance / np.log2(i + 2)  # log2(i+2) because rank is 1-indexed
    
    # IDCG: DCG if all relevant docs were retrieved first
    idcg = 0.0
    for i in range(min(len(relevant), k)):
        idcg += 1.0 / np.log2(i + 2)
    
    return dcg / idcg if idcg > 0 else 0.0

# Example: evaluate retrieval quality
# Assume a system retrieved these docs for a query
retrieved = ['doc1', 'doc3', 'doc5', 'doc2', 'doc7', 'doc9']
retrieved_with_scores = [('doc1', 0.95), ('doc3', 0.87), ('doc5', 0.82), ('doc2', 0.78), ('doc7', 0.65), ('doc9', 0.61)]
relevant = {'doc2', 'doc3', 'doc5'}  # Ground truth relevant docs

print("Example Query Evaluation:")
print(f"Retrieved (in order): {retrieved}")
print(f"Relevant (ground truth): {relevant}")
print()

# Calculate metrics for different k values
print("Metrics at different k:")
print("k\tP@k\tR@k\tMRR\tNDCG@k")
print("-" * 40)
for k in [1, 3, 5, 6]:
    p_at_k = precision_at_k(retrieved, relevant, k)
    r_at_k = recall_at_k(retrieved, relevant, k)
    mrr = mean_reciprocal_rank(retrieved, relevant) if k >= 1 else 0
    ndcg = ndcg_at_k(retrieved_with_scores, relevant, k)
    print(f"{k}\t{p_at_k:.2f}\t{r_at_k:.2f}\t{mrr:.2f}\t{ndcg:.3f}")

print()
print("Interpretation:")
print(f"  Precision@3 = {precision_at_k(retrieved, relevant, 3):.2f}: 2 out of 3 retrieved are relevant")
print(f"  Recall@3 = {recall_at_k(retrieved, relevant, 3):.2f}: retrieved 2 out of 3 relevant docs")
print(f"  MRR = {mean_reciprocal_rank(retrieved, relevant):.2f}: first relevant doc at rank 2")
print(f"  NDCG@5 = {ndcg_at_k(retrieved_with_scores, relevant, 5):.3f}: ranking quality (penalizes bad order)")

# Visualize metrics
k_values = [1, 2, 3, 4, 5, 6]
p_values = [precision_at_k(retrieved, relevant, k) for k in k_values]
r_values = [recall_at_k(retrieved, relevant, k) for k in k_values]
ndcg_values = [ndcg_at_k(retrieved_with_scores, relevant, k) for k in k_values]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, p_values, marker='o', label='Precision@k', linewidth=2, markersize=8)
ax.plot(k_values, r_values, marker='s', label='Recall@k', linewidth=2, markersize=8)
ax.plot(k_values, ndcg_values, marker='^', label='NDCG@k', linewidth=2, markersize=8)
ax.set_xlabel('k (number of retrieved documents)', fontsize=10)
ax.set_ylabel('Score', fontsize=10)
ax.set_title('Retrieval Evaluation Metrics', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()

## Section 8: Interview Takeaways

### Key Concepts

1. **RAG Pipeline**
   - **Retrieve**: Use semantic search to find relevant documents from a knowledge base
   - **Augment**: Inject retrieved context into the prompt
   - **Generate**: LLM answers based on grounded context, reducing hallucination
   - Advantage: More factually accurate than generation from LLM knowledge alone

2. **Vector Databases & Embeddings**
   - **Embeddings**: Dense vectors representing semantic meaning (from sentence-transformers, OpenAI, etc.)
   - **Cosine Similarity**: Standard metric for measuring document similarity
   - **Vector DBs**: Pinecone, Weaviate, Milvus, Chroma, FAISS enable fast similarity search at scale
   - **FAISS**: Facebook's library for efficient similarity search and clustering

3. **Chunking Strategies**
   - **Fixed-size**: Simple, but may split sentences awkwardly
   - **Semantic**: Group by meaning (better quality, but slower)
   - **Overlap**: Prevents loss of context at chunk boundaries
   - Trade-off: chunk size affects retrieval quality and computational cost

4. **Approximate Nearest Neighbor (ANN) Search**
   - **Exact search** (IndexFlatIP): O(n) but guaranteed correct, slow for millions of vectors
   - **Approximate** (IndexIVFFlat, HNSW, LSH): Sublinear, tunable accuracy-speed tradeoff
   - For production: use approximate methods with careful recall tuning

5. **Evaluation Metrics for Retrieval**
   - **Precision@k**: Fraction of retrieved docs that are relevant → focus on quality
   - **Recall@k**: Fraction of relevant docs that are retrieved → focus on coverage
   - **MRR**: Reciprocal rank of first relevant result → measures ranking quality
   - **NDCG@k**: Normalized discounted cumulative gain → accounts for ranking order
   - All metrics important: high recall without precision wastes tokens, high precision without recall misses answers

6. **Embedding Model Choice**
   - Model quality directly impacts retrieval quality
   - Sentence-transformers: lightweight, fast (all-MiniLM-L6-v2 used here)
   - OpenAI embeddings: more expensive, potentially better semantic understanding
   - Domain-specific embeddings: fine-tune for specialized tasks
   - Choice of embedding model can be as critical as LLM choice

### Common Interview Questions

**Q: How does RAG differ from fine-tuning?**  
A: RAG retrieves external context at inference time (dynamic, no retraining). Fine-tuning updates model weights (static knowledge, requires retraining). RAG is better for rapidly changing data; fine-tuning for stable, domain-specific patterns.

**Q: Why normalize embeddings for cosine similarity in FAISS?**  
A: FAISS IndexFlatIP computes inner product. For normalized vectors, inner product equals cosine similarity. Normalization ensures all vectors have unit length.

**Q: What's the impact of chunk size?**  
A: Larger chunks provide more context but may dilute relevance signals. Smaller chunks are more targeted but may lose coherence. Typical: 256-512 tokens with 10-20% overlap.

**Q: How do you evaluate RAG quality?**  
A: Measure retrieval quality (Precision, Recall, NDCG) separately from generation quality. Use RAGAS or similar frameworks to assess end-to-end factuality.

**Q: Exact vs approximate search — when to use each?**  
A: Exact for small/interactive datasets (<1M vectors). Approximate for production/large scale (recall tuning essential).

**Q: How do you handle domain-specific queries?**  
A: Use domain-trained embeddings, filter by metadata, use rerankers, hybrid search (sparse + dense).

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>